# Hybrid SNR-aware Attention

This notebook evaluates an inference-time hybrid of two trained models:

- use plain differential attention for low/transition SNR: `-8 dB` to `+2 dB`
- use normal attention outside that range

The goal is to test the research claim:

> Differential attention is useful mainly in low SNR, while normal attention remains stronger at cleaner SNR.

The final result is compared against the normal attention baseline and exported under:

```text
experiments/5class_hybrid_snr_aware_attention/
```


In [ ]:
# CELL 1: Setup repo and paths
import os
import sys
import shutil
import subprocess
from pathlib import Path
from datetime import datetime

import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, FileLink

os.environ["KERAS_BACKEND"] = "tensorflow"

REPO_URL = "https://github.com/akshlabh/amr-5-class.git"
WORK_DIR = Path("/kaggle/working/amr-5-class")
DATASET = Path("/kaggle/input/datasets/gustavopolicarpo/rml201610a-dict/RML2016.10a_dict.dat")


def find_attached_repo():
    for root in Path("/kaggle/input").glob("**"):
        if (root / "src" / "train.py").exists() and (root / "configs").exists():
            return root
    return None


if (WORK_DIR / ".git").exists():
    subprocess.run(["git", "-C", str(WORK_DIR), "pull"], check=False)
elif WORK_DIR.exists() and (WORK_DIR / "src" / "train.py").exists():
    print(f"Using existing work dir: {WORK_DIR}")
else:
    attached = find_attached_repo()
    if attached is not None:
        print(f"Copying attached repo from: {attached}")
        if WORK_DIR.exists():
            shutil.rmtree(WORK_DIR)
        shutil.copytree(attached, WORK_DIR)
    else:
        print("Cloning repo from GitHub...")
        subprocess.run(["git", "clone", REPO_URL, str(WORK_DIR)], check=True)

os.chdir(WORK_DIR)
if str(WORK_DIR) not in sys.path:
    sys.path.insert(0, str(WORK_DIR))

NORMAL_DIR = Path("experiments/5class_attention")
DIFF_DIR = Path("experiments/5class_diffattention")
HYBRID_DIR = Path("experiments/5class_hybrid_snr_aware_attention")

required = [
    "src/evaluate_hybrid_snr_aware_attention.py",
    "src/models/mcldnn_attention.py",
    "src/models/mcldnn_diffattention.py",
    "configs/exp_5class_attention.yaml",
    "configs/exp_5class_diffattention.yaml",
    "src/train.py",
]

print(f"Working dir: {Path.cwd()}")
print(f"Dataset    : {DATASET}")
print(f"Dataset OK : {DATASET.exists()}")
for f in required:
    print(f"{'OK' if Path(f).exists() else 'MISSING'} {f}")

assert DATASET.exists(), f"Dataset not found: {DATASET}"
for f in required:
    assert Path(f).exists(), f"Required repo file missing: {f}"


In [ ]:
# CELL 2: Ensure normal and plain differential attention checkpoints exist
# These are the two source models used by the hybrid.

baseline_jobs = [
    ("normal_attention", NORMAL_DIR, "configs/exp_5class_attention.yaml"),
    ("plain_diffattention", DIFF_DIR, "configs/exp_5class_diffattention.yaml"),
]

for name, exp_dir, cfg in baseline_jobs:
    weights = exp_dir / "checkpoints" / "best_model.weights.h5"
    score = exp_dir / "results" / "test_score.csv"
    snr = exp_dir / "results" / "acc_per_snr.csv"
    if weights.exists() and score.exists() and snr.exists():
        print(f"{name}: existing checkpoint/results found.")
    else:
        print(f"{name}: missing checkpoint/results. Training now...")
        subprocess.run([
            sys.executable, "src/train.py",
            "--config", cfg,
            "--datasetpath", str(DATASET),
        ], check=True)
    print(f"  weights={weights.exists()}  score={score.exists()}  snr={snr.exists()}")


In [ ]:
# CELL 3: Evaluate Hybrid SNR-aware Attention
# Default rule: use differential attention for -8 <= SNR <= +2, normal elsewhere.

if HYBRID_DIR.exists():
    print(f"Removing old hybrid outputs: {HYBRID_DIR}")
    shutil.rmtree(HYBRID_DIR)

process = subprocess.Popen(
    [
        sys.executable, "-u", "src/evaluate_hybrid_snr_aware_attention.py",
        "--datasetpath", str(DATASET),
        "--normal-weights", str(NORMAL_DIR / "checkpoints" / "best_model.weights.h5"),
        "--diff-weights", str(DIFF_DIR / "checkpoints" / "best_model.weights.h5"),
        "--output-dir", str(HYBRID_DIR),
        "--diff-snr-min", "-8",
        "--diff-snr-max", "2",
    ],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

for line in process.stdout:
    print(line, end="", flush=True)

return_code = process.wait()
if return_code != 0:
    raise subprocess.CalledProcessError(return_code, process.args)

print("Hybrid evaluation finished.")
print("Results:", (HYBRID_DIR / "results" / "test_score.csv").exists())
print("SNR CSV:", (HYBRID_DIR / "results" / "acc_per_snr.csv").exists())


In [ ]:
# CELL 4: Display key comparison tables and plots
score = pd.read_csv(HYBRID_DIR / "results" / "test_score.csv")
snr = pd.read_csv(HYBRID_DIR / "results" / "acc_per_snr.csv")

print("Test score summary:")
display(score)

print("Per-SNR hybrid vs normal:")
display(snr)

fig_path = HYBRID_DIR / "figures" / "hybrid_vs_normal_acc_vs_snr.png"
delta_path = HYBRID_DIR / "figures" / "hybrid_minus_normal_delta_by_snr.png"
print("Saved plots:")
print(fig_path)
print(delta_path)


In [ ]:
# CELL 5: Explicit Normal vs Hybrid comparison inside notebook
# This cell creates notebook-visible comparison tables and plots.

COMPARE_DIR = HYBRID_DIR / "results" / "comparison_with_normal_attention"
COMPARE_DIR.mkdir(parents=True, exist_ok=True)

score = pd.read_csv(HYBRID_DIR / "results" / "test_score.csv")
snr = pd.read_csv(HYBRID_DIR / "results" / "acc_per_snr.csv")

# Make the test summary easy to read.
test_summary = score.copy()
test_summary["accuracy_percent"] = 100.0 * test_summary["accuracy"]
if set(test_summary["model"]) >= {"normal_attention", "hybrid_snr_aware_attention"}:
    normal_acc = float(test_summary.loc[test_summary["model"] == "normal_attention", "accuracy"].iloc[0])
    hybrid_acc = float(test_summary.loc[test_summary["model"] == "hybrid_snr_aware_attention", "accuracy"].iloc[0])
    normal_loss = float(test_summary.loc[test_summary["model"] == "normal_attention", "loss"].iloc[0])
    hybrid_loss = float(test_summary.loc[test_summary["model"] == "hybrid_snr_aware_attention", "loss"].iloc[0])
else:
    raise ValueError("Expected normal_attention and hybrid_snr_aware_attention rows in test_score.csv")

headline = pd.DataFrame([
    {
        "comparison": "Hybrid - Normal",
        "normal_accuracy_percent": 100.0 * normal_acc,
        "hybrid_accuracy_percent": 100.0 * hybrid_acc,
        "delta_percent_points": 100.0 * (hybrid_acc - normal_acc),
        "normal_loss": normal_loss,
        "hybrid_loss": hybrid_loss,
    }
])

snr_compare = snr.copy()
snr_compare["normal_accuracy_percent"] = 100.0 * snr_compare["normal_attention_accuracy"]
snr_compare["hybrid_accuracy_percent"] = 100.0 * snr_compare["hybrid_accuracy"]
snr_compare["delta_percent_points"] = 100.0 * snr_compare["delta_hybrid_minus_normal"]

headline_csv = COMPARE_DIR / "normal_vs_hybrid_summary.csv"
snr_csv = COMPARE_DIR / "normal_vs_hybrid_acc_per_snr.csv"
headline.to_csv(headline_csv, index=False)
snr_compare.to_csv(snr_csv, index=False)

print("Normal vs Hybrid headline summary:")
display(headline)

print("Normal vs Hybrid by SNR:")
display(snr_compare[[
    "snr",
    "selected_model",
    "normal_accuracy_percent",
    "hybrid_accuracy_percent",
    "delta_percent_points",
]])

fig, ax = plt.subplots(figsize=(11, 6))
ax.plot(snr_compare["snr"], snr_compare["normal_accuracy_percent"],
        marker="o", linewidth=2.5, label="Normal attention baseline")
ax.plot(snr_compare["snr"], snr_compare["hybrid_accuracy_percent"],
        marker="s", linewidth=2.5, label="Hybrid SNR-aware attention")
ax.set_title("Normal Attention vs Hybrid SNR-aware Attention")
ax.set_xlabel("SNR (dB)")
ax.set_ylabel("Accuracy (%)")
ax.grid(True, alpha=0.3)
ax.legend()
ax.set_xticks(snr_compare["snr"])
plt.tight_layout()
curve_path = COMPARE_DIR / "normal_vs_hybrid_acc_vs_snr.png"
fig.savefig(curve_path, dpi=180, bbox_inches="tight")
plt.show()

fig, ax = plt.subplots(figsize=(11, 4.8))
colors = ["#2ca02c" if x >= 0 else "#d62728" for x in snr_compare["delta_percent_points"]]
ax.bar(snr_compare["snr"].astype(str), snr_compare["delta_percent_points"], color=colors)
ax.axhline(0, color="black", linewidth=1)
ax.set_title("Hybrid minus Normal Attention by SNR")
ax.set_xlabel("SNR (dB)")
ax.set_ylabel("Delta accuracy (percentage points)")
ax.grid(True, axis="y", alpha=0.3)
plt.tight_layout()
delta_path = COMPARE_DIR / "hybrid_minus_normal_delta_by_snr.png"
fig.savefig(delta_path, dpi=180, bbox_inches="tight")
plt.show()

print(f"Saved: {headline_csv}")
print(f"Saved: {snr_csv}")
print(f"Saved: {curve_path}")
print(f"Saved: {delta_path}")


In [ ]:
# CELL 6: Create repo-ready zip for ONLY hybrid results
stamp = datetime.now().strftime("%Y%m%d_%H%M")
zip_base = Path("/kaggle/working") / f"hybrid_snr_aware_attention_repo_ready_{stamp}"
zip_path = shutil.make_archive(
    str(zip_base),
    "zip",
    root_dir=str(WORK_DIR),
    base_dir="experiments/5class_hybrid_snr_aware_attention",
)

print(f"Created repo-ready zip: {zip_path}")
print("\nExtract this zip at the repo root. It will create/update:")
print("  experiments/5class_hybrid_snr_aware_attention/")
print("\nIncluded files:")
for path in sorted(HYBRID_DIR.rglob("*")):
    if path.is_file():
        print(" -", Path("experiments/5class_hybrid_snr_aware_attention") / path.relative_to(HYBRID_DIR))

display(FileLink(zip_path))
